In [ ]:
# [0 · Imports & configuration]
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sc.set_figure_params(dpi=100, frameon=False)
plt.rcParams['figure.max_open_warning'] = 0

print(f'scvi-tools: {scvi.__version__}')

from nalm_utils import *

CACHE_DIR        = Path('/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data/cache')
SOURCE_CACHE     = CACHE_DIR / 'adata_all_annotated.h5ad'
ANNOTATED_CACHE  = CACHE_DIR / 'adata_cytovi_annotated.h5ad'
MODEL_DIR        = CACHE_DIR / 'cytovi_model'
ASINH_SCALE      = 5.0
LEIDEN_RES       = 0.5
FDR_THRESH       = 0.05
spatial_rep      = 'spatial_asinh5_top500var'

In [ ]:
# [1 · Data loading & subsetting]
ANNOTATED_CACHE = CACHE_DIR / 'adata_cytovi_annotated_compat.h5ad'
adata = sc.read_h5ad(ANNOTATED_CACHE)

# Subset to 6h Mock, T cells only (CD4 + CD8), healthy-T systems only
mask_6h_mock_t = (
    (adata.obs['time'] == '6h') &
(adata.obs['condition'] == 'Mock') &
    (adata.obs['cell_type_annot'].isin(['CD4', 'CD8','B'])) &
    (adata.obs['cell_system'].isin(['NALM-6 + healthy T', 'healthy B + healthy T']))
)
adata_6h = adata[mask_6h_mock_t].copy()

print(f'6h Mock T cells (healthy-T systems): {adata_6h.n_obs}')
pd.crosstab(index=adata_6h.obs['sample'], columns=[adata_6h.obs['cell_system'], adata_6h.obs['cell_type_annot']])

# T ANALYSIS

## COMPARE 6h mock between healthy b and NALM6

In [ ]:
# [2 · UMAP overview]
sc.pl.umap(adata_6h,                                                       
  color=['CD3e','CD19','CD326','cell_system','cell_type_annot','condition'],          
  layer='arcsinh', frameon=False)


In [ ]:
# [2a · Raw vs arcsinh histograms for CD3e, CD19, CD20]
plot_raw_vs_arcsinh(
    adata_6h,
    markers=['CD3e', 'CD19', 'CD20'],
    group_key='cell_system',
    groups=[
        ('NALM-6 + healthy T', '#d62728'),
        ('healthy B + healthy T', '#1f77b4'),
    ],
)

In [ ]:
# [2a2 · CD19 arcsinh histogram — CD8 cells, both systems]
plot_raw_vs_arcsinh(
    adata_6h[adata_6h.obs['cell_type_annot'] == 'CD8'],
    markers=['CD19'],
    group_key='cell_system',
    groups=[
        ('NALM-6 + healthy T',   '#d62728'),
        ('healthy B + healthy T', '#1f77b4'),
    ],
)

In [ ]:
# [2b · B cell score]
B_CELL_MARKERS = load_marker_panel('b_cell_markers')
print(f'B cell markers ({len(B_CELL_MARKERS)}): {B_CELL_MARKERS}')

compute_b_cell_score(adata_6h, B_CELL_MARKERS)
sc.pl.umap(adata_6h, color=['b_cell_score', 'cell_type_annot', 'cell_system'], frameon=False)

In [ ]:
# [3 · Differential abundance (arcsinh)]
SYS_A = 'NALM-6 + healthy T'
SYS_B = 'healthy B + healthy T'

plot_da_bars(adata_6h, ['CD4', 'CD8'], SYS_A, SYS_B)

In [ ]:
# [4 · Colocalization setup + top partners (NALM-6 CD8)]
# ── Colocalization analysis: B-cell markers in CD8 T cells ──────────────────
B_MARKERS = ['CD20', 'CD22', 'CD37', 'CD32', 'CD35']

sp_full = adata_6h.obsm['spatial_asinh5']
if not isinstance(sp_full, pd.DataFrame):
    sp_full = pd.DataFrame(sp_full, index=adata_6h.obs_names)

nalm_idx    = (adata_6h.obs['cell_type_annot'] == 'CD8') & (adata_6h.obs['cell_system'] == SYS_A)
healthy_idx = (adata_6h.obs['cell_type_annot'] == 'CD8') & (adata_6h.obs['cell_system'] == SYS_B)

sp_cd8_nalm    = sp_full.loc[nalm_idx]
sp_cd8_healthy = sp_full.loc[healthy_idx]

print(f'CD8 NALM-6 cells: {nalm_idx.sum()}')
print(f'CD8 healthy B cells: {healthy_idx.sum()}')
print(f'Total colocalization pairs: {sp_full.shape[1]}')

plot_top_coloc_partners(sp_cd8_nalm, B_MARKERS, 'NALM-6 CD8', '#d62728')

In [ ]:
# [5 · Top colocalization partners (healthy B CD8)]
plot_top_coloc_partners(sp_cd8_healthy, B_MARKERS, 'healthy B CD8', '#1f77b4')

In [ ]:
# [6 · Differential colocalization (NALM-6 vs healthy B)]
plot_diff_coloc(sp_cd8_nalm, sp_cd8_healthy, B_MARKERS, sp_full.columns,
               sys_a_label='NALM-6+T', sys_b_label='Healthy B+T')

In [ ]:
# [7 · CD20 neighborhood: top 30 by mean colocalization]
# ── Top 30 CD20 colocalization partners from healthy B CD8 ──────────────────
top30_partners = select_top_partners(sp_cd8_healthy, 'CD20', sp_full.columns, n=30)
proteins_oi = set(['CD20'] + top30_partners)
oi_pair_cols = get_pairwise_cols(sp_full.columns, proteins_oi)
markers_sorted = sorted(proteins_oi)

print('Top 30 CD20 colocalization partners (healthy B CD8):')
for i, p in enumerate(top30_partners, 1):
    print(f'  {i:2d}. {p}')

mat_a    = build_mean_matrix(sp_cd8_healthy, oi_pair_cols, markers_sorted)
mat_b    = build_mean_matrix(sp_cd8_nalm,    oi_pair_cols, markers_sorted)
mat_diff = mat_b - mat_a
row_linkage = compute_ward_linkage(mat_diff)

plot_neighborhood_suite(
    [(mat_a, 'Healthy B CD8 (mean z)', 'RdBu_r'),
     (mat_b, 'NALM-6 CD8 (mean z)', 'RdBu_r'),
     (mat_diff, 'Diff (NALM-6 − Healthy)', 'coolwarm')],
    row_linkage,
    'CD20 + top 30 healthy B CD8 partners',
    cluster_matrices={'Healthy': mat_a},
    cluster_title='Healthy B CD8',
)

In [ ]:
# [8 · CD20 neighborhood: top 30 by significance]
# ── Top 30 most significantly different CD20 colocalization partners ─────────
cd20_cols = get_marker_cols(sp_full.columns, 'CD20')
diff_df = compute_marker_diff_coloc(sp_cd8_nalm, sp_cd8_healthy, cd20_cols, 'CD20')
top30_sig = diff_df.head(30)['partner'].tolist()

print('Top 30 most significantly different CD20 pairs (NALM-6 vs healthy B CD8):')
print(diff_df.head(30)[['partner', 'mean_a', 'mean_b', 'mean_diff', 'padj']].to_string(index=False))

proteins_sig = set(['CD20'] + top30_sig)
cols_sig = get_pairwise_cols(sp_full.columns, proteins_sig)
markers_sig = sorted(proteins_sig)

mat_n_sig = build_mean_matrix(sp_cd8_nalm, cols_sig, markers_sig)
mat_h_sig = build_mean_matrix(sp_cd8_healthy, cols_sig, markers_sig)
mat_diff_sig = mat_n_sig - mat_h_sig
linkage_sig = compute_ward_linkage(mat_diff_sig)

plot_neighborhood_suite(
    [(mat_n_sig, 'NALM-6 CD8 (mean z)', 'RdBu_r'),
     (mat_h_sig, 'Healthy B CD8 (mean z)', 'RdBu_r'),
     (mat_diff_sig, 'Diff (NALM-6 − Healthy)', 'coolwarm')],
    linkage_sig,
    'CD20 + top 30 most significantly different partners',
    cluster_matrices={'NALM-6': mat_n_sig, 'Healthy': mat_h_sig},
    cluster_title='Top 30 sig. diff CD20 partners',
)

In [ ]:
# [9 · DA without B cell markers (CD8 only)]
# Remove B cell markers from var_names to focus on T cell-intrinsic differences
b_set = set(B_CELL_MARKERS) & set(adata_6h.var_names)
print(f'Removing {len(b_set)} B cell markers: {sorted(b_set)}')

adata_6h_no_b = adata_6h[:, ~adata_6h.var_names.isin(b_set)].copy()
print(f'Remaining markers: {adata_6h_no_b.n_vars}')

plot_da_bars(adata_6h_no_b, ['CD8'], SYS_A, SYS_B)

In [ ]:
# [10 · CD326 (EpCAM) colocalization analysis]
sc.pl.umap(adata_6h, color=['CD3e', 'CD326', 'CD19'], layer='arcsinh', frameon=False)

# Top colocalization partners for CD326 in each system
plot_top_coloc_partners(sp_cd8_nalm, ['CD326'], 'NALM-6 CD8', '#d62728')
plot_top_coloc_partners(sp_cd8_healthy, ['CD326'], 'healthy B CD8', '#1f77b4')

# Differential colocalization
plot_diff_coloc(sp_cd8_nalm, sp_cd8_healthy, ['CD326'], sp_full.columns,
               sys_a_label='NALM-6+T', sys_b_label='Healthy B+T')

## Healthy T + Healthy B — 6h vs 48h, Mock vs Blinatumomab

In [ ]:
# [11 · Subset: healthy B+T, CD8, all timepoints & conditions]
mask_healthy_cd8 = (
    (adata.obs['cell_system'] == 'healthy B + healthy T') &
    (adata.obs['cell_type_annot'] == 'CD8')
)
adata_healthy_cd8 = adata[mask_healthy_cd8].copy()

# Create combined group label
adata_healthy_cd8.obs['time_cond'] = (
    adata_healthy_cd8.obs['time'].astype(str) + ' ' +
    adata_healthy_cd8.obs['condition'].astype(str)
)

print(f'CD8 cells in healthy B+T: {adata_healthy_cd8.n_obs}')
print(adata_healthy_cd8.obs['time_cond'].value_counts().to_string())

In [ ]:
# [12 · 4-way DA: CD8 healthy B+T vs NALM-6+T (6h/48h × Mock/Blina)]
B_CELL_MARKERS_SET = set(load_marker_panel('b_cell_markers'))

# Primary: CD8 in healthy B + healthy T
adata_healthy_cd8_no_b = adata_healthy_cd8[:, ~adata_healthy_cd8.var_names.isin(B_CELL_MARKERS_SET)].copy()

# Comparison: CD8 in NALM-6 + healthy T
mask_nalm_cd8 = (
    (adata.obs['cell_system'] == 'NALM-6 + healthy T') &
    (adata.obs['cell_type_annot'] == 'CD8')
)
adata_nalm_cd8 = adata[mask_nalm_cd8].copy()
adata_nalm_cd8.obs['time_cond'] = (
    adata_nalm_cd8.obs['time'].astype(str) + ' ' +
    adata_nalm_cd8.obs['condition'].astype(str)
)
adata_nalm_cd8_no_b = adata_nalm_cd8[:, ~adata_nalm_cd8.var_names.isin(B_CELL_MARKERS_SET)].copy()

print(f'CD8 healthy B+T: {adata_healthy_cd8_no_b.n_obs}')
print(f'CD8 NALM-6+T:    {adata_nalm_cd8_no_b.n_obs}')

plot_marker_panel_violins(
    adata_healthy_cd8_no_b, 'cd8_t_cell_markers',
    group_key='time_cond',
    adata_compare=adata_nalm_cd8_no_b,
    primary_label='Healthy B + T',
    compare_label='NALM-6 + T',
)

# co-conditions

In [ ]:
# [12b · LFC scatter — Blina vs Mock, NALM-6+T (x) vs Healthy B+T (y), CD8]
# Per-marker log2 fold change (Blina / Mock) for CD8 cells, one panel per timepoint.
# Markers farthest from the y=x diagonal are colored and printed.

def _cd8_lfc(adata_full, time_val, system_val, layer='raw', pseudo=1.0):
    mask = (
        (adata_full.obs['time'] == time_val) &
        (adata_full.obs['cell_type_annot'] == 'CD8') &
        (adata_full.obs['cell_system'] == system_val)
    )
    sub = adata_full[mask]
    X = np.array(sub.layers[layer], dtype=np.float32)
    mock  = X[(sub.obs['condition'] == 'Mock').values].mean(axis=0)
    blina = X[(sub.obs['condition'] == 'Blinatumomab').values].mean(axis=0)
    return pd.Series(np.log2((blina + pseudo) / (mock + pseudo)), index=sub.var_names)

adata_cd8 = adata[adata.obs['cell_type_annot'] == 'CD8'].copy()

SYS_N = 'NALM-6 + healthy T'
SYS_H = 'healthy B + healthy T'
TOP_K = 10  # markers farthest from diagonal to highlight/print

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, time_val in zip(axes, ['6h', '48h']):
    lfc_n = _cd8_lfc(adata_cd8, time_val, SYS_N)
    lfc_h = _cd8_lfc(adata_cd8, time_val, SYS_H)
    df_lfc = pd.DataFrame({'nalm': lfc_n, 'healthy': lfc_h}).replace(
        [np.inf, -np.inf], np.nan).dropna()

    # Signed perpendicular distance from y=x: positive = above diagonal (higher in healthy)
    df_lfc['off_diag'] = (df_lfc['healthy'] - df_lfc['nalm']) / np.sqrt(2)
    top_idx = df_lfc['off_diag'].abs().nlargest(TOP_K).index
    colors = np.where(df_lfc.loc[top_idx, 'off_diag'] > 0, '#1f77b4', '#d62728')
    color_map = dict(zip(top_idx, colors))

    lim = max(df_lfc[['nalm', 'healthy']].abs().max().max() * 1.15, 0.1)
    ax.axhline(0, color='grey', lw=0.7, ls='--')
    ax.axvline(0, color='grey', lw=0.7, ls='--')
    ax.plot([-lim, lim], [-lim, lim], color='grey', lw=0.7, ls=':')

    other_idx = df_lfc.index.difference(top_idx)
    ax.scatter(df_lfc.loc[other_idx, 'nalm'], df_lfc.loc[other_idx, 'healthy'],
               s=30, alpha=0.55, color='#bbbbbb', edgecolor='white', linewidth=0.5)
    ax.scatter(df_lfc.loc[top_idx, 'nalm'], df_lfc.loc[top_idx, 'healthy'],
               s=70, alpha=0.9, c=[color_map[m] for m in top_idx],
               edgecolor='black', linewidth=0.6, zorder=3)

    for m in top_idx:
        ax.annotate(display_name(m),
                    (df_lfc.loc[m, 'nalm'], df_lfc.loc[m, 'healthy']),
                    fontsize=9, fontweight='bold', color=color_map[m],
                    xytext=(4, 4), textcoords='offset points')

    r = df_lfc['nalm'].corr(df_lfc['healthy'])
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.set_xlabel(f'LFC Blina vs Mock\n{SYS_N}')
    ax.set_ylabel(f'LFC Blina vs Mock\n{SYS_H}')
    ax.set_title(f'CD8  —  {time_val}   (n={len(df_lfc)}, Pearson r={r:.2f})')

    # Print top markers farthest from diagonal
    print(f'\n{"=" * 65}')
    print(f'  CD8 {time_val}  —  top {TOP_K} markers farthest from y=x')
    print(f'  blue = higher LFC in {SYS_H} | red = higher LFC in {SYS_N}')
    print(f'{"=" * 65}')
    top_tbl = df_lfc.loc[top_idx, ['nalm', 'healthy', 'off_diag']].copy()
    top_tbl.index = [display_name(m) for m in top_tbl.index]
    top_tbl.columns = ['LFC_NALM', 'LFC_Healthy', 'off_diag']
    top_tbl = top_tbl.reindex(top_tbl['off_diag'].abs().sort_values(ascending=False).index)
    print(top_tbl.round(3).to_string())

plt.tight_layout()
plt.show()

In [ ]:
# [12c · NALM-6 + healthy T, CD8 — Mock vs Blina: abundance + spatial, per timepoint]
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

SYS_N = 'NALM-6 + healthy T'
TOP_N = 15


def _mw_compare(values_a, values_b, labels):
    """Per-feature MW U test. mean_diff = mean(a) - mean(b); positive = higher in a."""
    rows = []
    for j, name in enumerate(labels):
        a, b = values_a[:, j], values_b[:, j]
        mean_diff = a.mean() - b.mean()
        _, p = mannwhitneyu(a, b, alternative='two-sided')
        rows.append({'feature': name, 'mean_diff': mean_diff, 'pval': p})
    df = pd.DataFrame(rows)
    _, df['padj'], _, _ = multipletests(df['pval'], method='fdr_bh')
    return df


def _cd8_subset(adata_full, time_val, sys_val):
    mask = (
        (adata_full.obs['time'] == time_val) &
        (adata_full.obs['cell_type_annot'] == 'CD8') &
        (adata_full.obs['cell_system'] == sys_val)
    )
    return adata_full[mask]


def _blina_vs_mock(sub, layer='arcsinh'):
    cond = sub.obs['condition'].values
    X = np.array(sub.layers[layer], dtype=np.float32)
    return _mw_compare(X[cond == 'Blinatumomab'], X[cond == 'Mock'], list(sub.var_names))


def _blina_vs_mock_spatial(sub, obsm_key='spatial_asinh5'):
    sp = sub.obsm[obsm_key]
    if not isinstance(sp, pd.DataFrame):
        sp = pd.DataFrame(sp, index=sub.obs_names)
    cond = sub.obs['condition'].values
    vals = sp.values
    return _mw_compare(vals[cond == 'Blinatumomab'], vals[cond == 'Mock'], list(sp.columns))


def _plot_bars(ax, top, color, xlabel, title, label_fmt=display_name, tick_fs=8):
    labels = [label_fmt(f) for f in top['feature']]
    ax.barh(labels, top['mean_diff'], color=color, alpha=0.85)
    ax.axvline(0, color='black', lw=0.7, ls='--')
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    ax.tick_params(axis='y', labelsize=tick_fs)
    xrng = max(top['mean_diff'].abs().max(), 1e-6)
    pad = xrng * 0.02
    for i, (_, r) in enumerate(top.iterrows()):
        s = sig_label(r['padj'])
        xp = r['mean_diff'] + (pad if r['mean_diff'] >= 0 else -pad)
        ha = 'left' if r['mean_diff'] >= 0 else 'right'
        ax.text(xp, i, s, va='center', ha=ha, fontsize=7)


def _pair_label(pair_name):
    a, b = split_pair(pair_name)
    return f'{display_name(a)} / {display_name(b)}'


for time_val in ['6h', '48h']:
    sub = _cd8_subset(adata, time_val, SYS_N)
    n_mock  = (sub.obs['condition'] == 'Mock').sum()
    n_blina = (sub.obs['condition'] == 'Blinatumomab').sum()
    print(f'\n[{time_val}] CD8 in {SYS_N}: Mock n={n_mock}, Blina n={n_blina}')

    da_ab = _blina_vs_mock(sub)
    da_sp = _blina_vs_mock_spatial(sub)

    fig, axes = plt.subplots(2, 2, figsize=(16, 13))

    # Row 0 — abundance
    top_up_ab = da_ab.nlargest(TOP_N, 'mean_diff').sort_values('mean_diff')
    top_dn_ab = da_ab.nsmallest(TOP_N, 'mean_diff').sort_values('mean_diff')
    _plot_bars(axes[0, 0], top_dn_ab, '#1f77b4',
               'Mean diff (arcsinh, Blina − Mock)',
               f'Abundance — top {TOP_N} higher in Mock')
    _plot_bars(axes[0, 1], top_up_ab, '#d62728',
               'Mean diff (arcsinh, Blina − Mock)',
               f'Abundance — top {TOP_N} higher in Blina')

    # Row 1 — spatial colocalization
    top_up_sp = da_sp.nlargest(TOP_N, 'mean_diff').sort_values('mean_diff')
    top_dn_sp = da_sp.nsmallest(TOP_N, 'mean_diff').sort_values('mean_diff')
    _plot_bars(axes[1, 0], top_dn_sp, '#1f77b4',
               'Mean diff (Blina − Mock)',
               f'Spatial coloc — top {TOP_N} pairs higher in Mock',
               label_fmt=_pair_label, tick_fs=7)
    _plot_bars(axes[1, 1], top_up_sp, '#d62728',
               'Mean diff (Blina − Mock)',
               f'Spatial coloc — top {TOP_N} pairs higher in Blina',
               label_fmt=_pair_label, tick_fs=7)

    fig.suptitle(f'CD8 — {SYS_N} — {time_val}  (Blina vs Mock)',
                 fontsize=14, y=1.00)
    plt.tight_layout()
    plt.show()

    print(f'\n--- {time_val} abundance (top higher in Blina) ---')
    out = top_up_ab.iloc[::-1][['feature', 'mean_diff', 'padj']].copy()
    out['feature'] = out['feature'].map(display_name)
    print(out.to_string(index=False))
    print(f'\n--- {time_val} spatial coloc (top higher in Blina) ---')
    out_sp = top_up_sp.iloc[::-1][['feature', 'mean_diff', 'padj']].copy()
    out_sp['feature'] = out_sp['feature'].map(_pair_label)
    print(out_sp.to_string(index=False))

## SPATIAL ANALYSIS FOR SELECTED MARKERS

In [ ]:
# [13 · Spatial subsets for selected-marker comparisons]
# Build spatial colocalization DataFrames for each condition × system × CD8

def _sp_subset(adata_full, time_val, cond_val, system_val):
    """Return spatial_asinh5 DataFrame for CD8 cells matching filters."""
    mask = (
        (adata_full.obs['time'] == time_val) &
        (adata_full.obs['condition'] == cond_val) &
        (adata_full.obs['cell_type_annot'] == 'CD8') &
        (adata_full.obs['cell_system'] == system_val)
    )
    sp = adata_full[mask].obsm['spatial_asinh5']
    if not isinstance(sp, pd.DataFrame):
        sp = pd.DataFrame(sp, index=adata_full.obs_names[mask])
    print(f'  {time_val} {cond_val:15s} {system_val:25s} → {sp.shape[0]} cells')
    return sp

SYS_H = 'healthy B + healthy T'
SYS_N = 'NALM-6 + healthy T'

print('Building spatial subsets (CD8 only):')
sp_6h_mock_h  = _sp_subset(adata, '6h',  'Mock',          SYS_H)
sp_6h_mock_n  = _sp_subset(adata, '6h',  'Mock',          SYS_N)
sp_6h_blina_h = _sp_subset(adata, '6h',  'Blinatumomab',  SYS_H)
sp_6h_blina_n = _sp_subset(adata, '6h',  'Blinatumomab',  SYS_N)
sp_48h_blina_h = _sp_subset(adata, '48h', 'Blinatumomab', SYS_H)
sp_48h_blina_n = _sp_subset(adata, '48h', 'Blinatumomab', SYS_N)

all_sp_cols = sp_6h_mock_h.columns

In [ ]:
# [25 · CD11a neighborhood heatmaps + networks: 3 conditions]
# ── Combine top 15 partners from each condition into a unified marker set ────
MARKER = 'CD11a'

top15_6h_n  = select_top_partners(sp_6h_blina_n,  MARKER, all_sp_cols, n=15)
top15_48h_n = select_top_partners(sp_48h_blina_n, MARKER, all_sp_cols, n=15)
top15_48h_h = select_top_partners(sp_48h_blina_h, MARKER, all_sp_cols, n=15)

combined_markers = sorted(set([MARKER] + top15_6h_n + top15_48h_n + top15_48h_h))
print(f'Combined marker set ({len(combined_markers)}): {[display_name(m) for m in combined_markers]}')

# ── Build mean colocalization matrices ───────────────────────────────────────
pair_cols = get_pairwise_cols(all_sp_cols, set(combined_markers))

mat_6h_n  = build_mean_matrix(sp_6h_blina_n,  pair_cols, combined_markers)
mat_48h_n = build_mean_matrix(sp_48h_blina_n, pair_cols, combined_markers)
mat_48h_h = build_mean_matrix(sp_48h_blina_h, pair_cols, combined_markers)

# ── Shared linkage from mean of all 3 matrices ──────────────────────────────
mat_avg = (mat_6h_n + mat_48h_n + mat_48h_h) / 3
row_linkage = compute_ward_linkage(mat_avg)

# ── Shared color scale ──────────────────────────────────────────────────────
vmax = max(mat_6h_n.abs().max().max(),
           mat_48h_n.abs().max().max(),
           mat_48h_h.abs().max().max())

n_markers = len(combined_markers)
fig_size = max(10, n_markers * 0.22)
tick_fs = max(5, min(8, 200 // n_markers))
subtitle = f'union of top 15 partners from 3 conditions, n={len(combined_markers)}'

for mat, label in [
    (mat_6h_n,  'NALM-6 6h Blina'),
    (mat_48h_n, 'NALM-6 48h Blina'),
    (mat_48h_h, 'Healthy 48h Blina'),
]:
    dn_map = {m: display_name(m) for m in mat.index}
    mat_disp = mat.rename(index=dn_map, columns=dn_map)

    # Heatmap
    g = sns.clustermap(
        mat_disp, cmap='RdBu_r', center=0, vmin=-vmax, vmax=vmax,
        row_linkage=row_linkage, col_linkage=row_linkage,
        figsize=(fig_size, fig_size), linewidths=0,
        xticklabels=True, yticklabels=True,
        cbar_kws={'shrink': 0.4, 'label': 'mean z-score'},
        dendrogram_ratio=0.08,
        cbar_pos=(0.02, 0.82, 0.03, 0.15),
    )
    g.ax_heatmap.tick_params(axis='both', labelsize=tick_fs)
    g.fig.suptitle(
        f'{display_name(MARKER)} neighborhood — {label}\n({subtitle})',
        fontsize=12, y=1.01,
    )
    plt.show()

    # Network graph
    fig_net, ax_net = plt.subplots(figsize=(10, 10))
    draw_force_net(
        mat,
        f'{display_name(MARKER)} neighborhood network — {label}\n({subtitle})',
        ax=ax_net, top_n=60, highlight_node=MARKER,
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# [26 · CD11a neighborhood: difference network 48h vs 6h Blina NALM-6]
mat_diff = mat_48h_n - mat_6h_n

fig_diff, ax_diff = plt.subplots(figsize=(10, 10))
draw_force_net(
    mat_diff,
    f'{display_name(MARKER)} neighborhood difference — NALM-6 48h minus 6h Blina\n({subtitle})',
    ax=ax_diff, top_n=60, highlight_node=MARKER,
)
plt.tight_layout()
plt.show()

In [ ]:
# [27 - CD8 immune synapse neighborhood: NALM-6 vs Healthy 6h Blina]
import matplotlib.patches as mpatches

SYNAPSE_CATEGORIES = {
    'cSMAC (signaling core)':  (['CD3e', 'CD8', 'CD2', 'CD28', 'CD134', 'CD137',
                                 'CD226', 'TIGIT', 'CD279', 'VISTA'],       '#e41a1c'),
    'pSMAC (adhesion ring)':   (['CD11a', 'CD50', 'KLRG1', 'CD94', 'CD48',
                                 'CD352', 'CD53'],                           '#4daf4a'),
    'Exclusion zone':          (['CD45', 'CD43', 'CD44'],                    '#377eb8'),
}

SYNAPSE_MARKERS = sorted(
    [m for markers, _ in SYNAPSE_CATEGORIES.values() for m in markers]
)
node_cmap = {}
for markers, color in SYNAPSE_CATEGORIES.values():
    for m in markers:
        node_cmap[m] = color

print(f'Synapse marker set ({len(SYNAPSE_MARKERS)}):')
for cat, (markers, color) in SYNAPSE_CATEGORIES.items():
    print(f'  {cat}: {[display_name(m) for m in markers]}')

syn_pair_cols = get_pairwise_cols(all_sp_cols, set(SYNAPSE_MARKERS))

mat_syn_n = build_mean_matrix(sp_6h_blina_n, syn_pair_cols, SYNAPSE_MARKERS)
mat_syn_h = build_mean_matrix(sp_6h_blina_h, syn_pair_cols, SYNAPSE_MARKERS)
mat_syn_diff = mat_syn_n - mat_syn_h

row_linkage_syn = compute_ward_linkage(mat_syn_diff)

cl = {cat: color for cat, (_, color) in SYNAPSE_CATEGORIES.items()}
for mm in [mat_syn_n, mat_syn_h, mat_syn_diff]:
    mm._node_category_labels = cl

vmax_syn = max(mat_syn_n.abs().max().max(), mat_syn_h.abs().max().max())
vmax_diff_syn = mat_syn_diff.abs().max().max()

n_syn = len(SYNAPSE_MARKERS)
fig_sz = max(10, n_syn * 0.28)
tk_fs = max(5, min(8, 200 // n_syn))
syn_sub = f'CD8 immune synapse markers (n={n_syn})'

configs = [
    (mat_syn_n,    'NALM-6 6h Blina',          'RdBu_r',  vmax_syn),
    (mat_syn_h,    'Healthy 6h Blina',          'RdBu_r',  vmax_syn),
    (mat_syn_diff, 'Diff (NALM-6 - Healthy)',   'coolwarm', vmax_diff_syn),
]

cat_handles = [mpatches.Patch(color=c, label=cat)
               for cat, (_, c) in SYNAPSE_CATEGORIES.items()]

for mat, label, cm, vm in configs:
    dn = {m: display_name(m) for m in mat.index}
    md = mat.rename(index=dn, columns=dn)
    rc = pd.Series({display_name(m): node_cmap.get(m, '#555555') for m in mat.index}, name='Category')

    g = sns.clustermap(
        md, cmap=cm, center=0, vmin=-vm, vmax=vm,
        row_linkage=row_linkage_syn, col_linkage=row_linkage_syn,
        row_colors=rc, col_colors=rc,
        figsize=(fig_sz, fig_sz), linewidths=0,
        xticklabels=True, yticklabels=True,
        cbar_kws={'shrink': 0.4, 'label': label},
        dendrogram_ratio=0.08, cbar_pos=(0.02, 0.82, 0.03, 0.15),
    )
    g.ax_heatmap.tick_params(axis='both', labelsize=tk_fs)
    g.ax_heatmap.legend(handles=cat_handles, loc='upper left', fontsize=7,
                        framealpha=0.9, title='Category', title_fontsize=8,
                        bbox_to_anchor=(-0.3, 1.0))
    g.fig.suptitle(f'Immune synapse - {label}\n({syn_sub})', fontsize=12, y=1.01)
    plt.show()

    fig_net, ax_net = plt.subplots(figsize=(10, 10))
    draw_force_net(mat, f'Immune synapse network - {label}\n({syn_sub})',
                   ax=ax_net, top_n=80, highlight_node='CD3e',
                   node_color_map=node_cmap, layout='kamada_kawai')
    plt.tight_layout()
    plt.show()